# Comment Classification: LLM Selection & Evaluation

Before running this notebook:
1. Install Ollama: https://ollama.com
2. To pull the models, run in terminal: 
    - ollama pull qwen3:8b
    - ollama pull llama3.1:8b

In [98]:
# importing libraries

import pandas as pd
import numpy as np
import ollama
from ollama import Client
import json
import re
from tqdm import tqdm
import krippendorff
from sklearn.metrics import classification_report, cohen_kappa_score

import ssl, certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/yuginkha/nltk_data...


In [99]:
data_comments_en = pd.read_csv("data/comments/comments_en.csv")

data_comments_en["scores"] = data_comments_en["raw_en"].apply(analyzer.polarity_scores)
data_comments_en["compound"] = data_comments_en["scores"].apply(lambda s: s["compound"])
data_comments_en["pos"] = data_comments_en["scores"].apply(lambda s: s["pos"])
data_comments_en["neg"] = data_comments_en["scores"].apply(lambda s: s["neg"])

def map_valence(row):
    if row["pos"] >= 0.15 and row["neg"] >= 0.15:
        return 3  # ambivalent
    elif row["compound"] >= 0.05:
        return 1  # positive
    elif row["compound"] <= -0.05:
        return 2  # negative
    else:
        return 4  # indifferent

data_comments_en["valence_vader"] = data_comments_en.apply(map_valence, axis=1)

In [100]:
merged_vader = data_eval.merge(data_comments_en[["row_id", "valence_vader"]], on="row_id", how="inner")

print(classification_report(merged_vader["valence"], merged_vader["valence_vader"]))
print("Cohen's kappa:", cohen_kappa_score(merged_vader["valence"], merged_vader["valence_vader"]))

              precision    recall  f1-score   support

           1       0.26      0.75      0.39        16
           2       0.57      0.65      0.61        62
           3       0.00      0.00      0.00        39
           4       0.24      0.22      0.23        18

    accuracy                           0.41       135
   macro avg       0.27      0.40      0.31       135
weighted avg       0.32      0.41      0.35       135

Cohen's kappa: 0.16451233842538193


In [73]:
# loading the complete comments dataset
data_cpath = "data/comments/comments.csv"
data_comments = pd.read_csv(data_cpath, encoding="latin1")

# loading manually coded dataset for evaluation
data_epath = "data/comments/comments_eval.xlsx"
data_eval = pd.read_excel(data_epath, engine="openpyxl")
PROMPT_IDS = [2, 3, 6, 20, 27, 28, 31, 32, 34, 57, 98, 99, 104, 111, 129]
data_eval = data_eval[~data_eval["unique_id"].isin(PROMPT_IDS)].copy()
print(f"{len(data_eval)} Zeilen im Testset")


135 Zeilen im Testset


Here, we collapse the `breadth` variable.  

In [74]:
data_eval["breadth"] = data_eval["breadth"].clip(upper=3)

In [75]:
# defining the system prompt inline with the codebook

with open("data/comments/system_prompt.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

In [76]:
# sanity check: kommt der Systemprompt beim Modell an?
# sanity check
print(len(SYSTEM_PROMPT))
print("LÄNGE IST KEIN BEWEIS" in SYSTEM_PROMPT)
print(SYSTEM_PROMPT[:300])


29830
True
Du kodierst deutschsprachige Nutzerkommentare für eine sozialwissenschaftliche Studie mit
dem Titel "Likes or Dislikes, Gratifications or Concerns?", zum deutschen 9-Punkte-Plan
gegen Terrorismus ("9-Punkte-Plan") bzw. zur Terrorismuspolitik allgemein. Die Analyseeinheit
ist ein einzelner Nutzerkomm


In [77]:
# define classifier function with qwen3:8b set as default

from ollama import Client
client = Client(timeout=120)
def classify_comment(text, parent_text=None, model="qwen3:8b"):
    """Classify a single comment on all 6 codebook variables in one call.
    Returns a dict of 6 ints, or a dict of -1s if parsing fails."""
    fallback = {"pers_exp": -1, "emot_exp": -1, "pol_opin": -1,
                "breadth": -1, "valence": -1, "contr": -1}

    if not isinstance(text, str) or text.strip() == "":
        return {"pers_exp": 0, "emot_exp": 0, "pol_opin": 0,
                "breadth": 0, "valence": 4, "contr": 0}

    user_msg = f"Comment: {text}"
    if parent_text:
        user_msg += f"\n\n(This is a reply to the following parent comment: {parent_text})"

    try:
        response = client.chat(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg}
            ],
            format="json",
            think=False,
            keep_alive="30m",
            options={"temperature": 0, "num_ctx": 8192, "num_predict": 400}
        )
    except Exception as e:
        print(f"Fehler: {e}")
        return fallback

    out = response["message"]["content"].strip()

    match = re.search(r"\{.*\}", out, re.DOTALL)
    if not match:
        return fallback

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return fallback

    result = {}
    bounds = {"pers_exp": (0,1), "emot_exp": (0,1), "pol_opin": (0,1),
              "breadth": (0,3), "valence": (1,4), "contr": (0,1)}
    for key, (lo, hi) in bounds.items():
        val = parsed.get(key, -1)
        try:
            val = int(val)
        except (TypeError, ValueError):
            val = -1
        result[key] = val if lo <= val <= hi else -1

    return result

In [78]:
# post_number resets per topic, so it's not unique on its own - build a composite key instead
def make_id(topic, post_number):
    return f"{topic}_{int(post_number)}"

# build ids on the FULL data_comments first, so parent lookups still work even for
# replies whose parent falls outside whatever subset we classify below
data_comments["row_id"] = data_comments.apply(lambda r: make_id(r["topic"], r["post_number"]), axis=1)
data_eval["row_id"] = data_eval.apply(lambda r: make_id(r["topic"], r["post_number"]), axis=1)
id_col = "row_id"

reply_col = "reply_to_post_number" if "reply_to_post_number" in data_comments.columns else None
text_by_id = dict(zip(data_comments["row_id"], data_comments["raw"].fillna("")))

def get_parent_text(row):
    if reply_col and pd.notna(row.get(reply_col)):
        parent_id = make_id(row["topic"], row[reply_col])
        return text_by_id.get(parent_id)
    return None

data_comments_full = data_comments.copy()  # keep the unfiltered version for the full run later

# for now, only classify comments that already have a manual code (validation pass) -
# comment this line out later to run the full dataset instead
data_comments = data_comments[data_comments[id_col].isin(data_eval[id_col])].copy()
print(f"Classifying {len(data_comments)} comments")

Classifying 135 comments


In [79]:
MODELS = ["qwen3.8:27b-q4_K_M"]

In [ ]:
VARS = ["pers_exp", "emot_exp", "pol_opin", "breadth", "valence", "contr"]
texts = data_comments["raw"].fillna("")


# delete:
#data_comments = data_comments.head().copy()


for model_name in MODELS:
    results = []
    for i, row in tqdm(data_comments.iterrows(), total=len(data_comments), desc=f"Classifying ({model_name})"):
        parent_text = get_parent_text(row)
        codes = classify_comment(texts.loc[i], parent_text=parent_text, model=model_name)
        results.append(codes)

    results_df = pd.DataFrame(results)
    model_tag = model_name.replace(":", "_").replace(".", "_")
    pred_cols = [f"{v}_{model_tag}" for v in VARS]
    data_comments[pred_cols] = results_df.values

Classifying (qwen3.8:27b-q4_K_M):   0%|          | 0/135 [00:30<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
client = Client(timeout=300)

r = client.chat(
    model="qwen3.8:27b-q4_K_M",
    messages=[{"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user", "content": "Comment: Heisse Luft, wie Alles, was von Merkel kommt."}],
    format="json",
    think=False,
    options={"temperature": 0, "num_ctx": 16384, "num_predict": 400}
)
print(repr(r["message"]["content"]))

'{"pers_exp": 0, "emot_exp": 0, "pol_opin": 1, "breadth": 1, "valence": 2, "contr": 1}'


In [ ]:
for tag in [model_tag]:
    cols = [f"{v}_{tag}" for v in VARS]
    print(tag, (data_comments[cols] == -1).sum().sum())
print(data_comments[[f"{v}_{model_tag}" for v in VARS]].head(10))

qwen3_8_27b-q4_K_M 42
     pers_exp_qwen3_8_27b-q4_K_M  emot_exp_qwen3_8_27b-q4_K_M  \
12                            -1                           -1   
29                             0                            1   
49                             0                            0   
57                            -1                           -1   
61                             0                            1   
66                             0                            1   
69                             0                            0   
71                             0                            1   
79                             0                            0   
100                            0                            0   

     pol_opin_qwen3_8_27b-q4_K_M  breadth_qwen3_8_27b-q4_K_M  \
12                            -1                          -1   
29                             1                           2   
49                             1                           2   
57    

In [ ]:
# create report

VARS = ["pers_exp", "emot_exp", "pol_opin", "breadth", "valence", "contr"]
var_weights = {"pers_exp": None, "emot_exp": None, "pol_opin": None, "contr": None,
               "breadth": "linear", "valence": None}
var_levels = {"pers_exp": "nominal", "emot_exp": "nominal", "pol_opin": "nominal",
              "contr": "nominal", "breadth": "ordinal", "valence": "nominal"}

def krippendorff_alpha(y_true, y_pred, level):
    try:
        data = np.array([y_true, y_pred], dtype=float)
        return krippendorff.alpha(reliability_data=data, level_of_measurement=level)
    except (ZeroDivisionError, ValueError):
        return float("nan")

data_eval_renamed = data_eval.rename(columns={v: f"{v}_true" for v in VARS})
merged = data_eval_renamed.merge(data_comments, on=id_col)
print(f"Matched {len(merged)} of {len(data_eval)} rows\n")

all_rows = []

for model_name in MODELS:
    model_tag = model_name.replace(":", "_").replace(".", "_")
    print(f"\n########## {model_name} ##########")
    for var in VARS:
        y_true = merged[f"{var}_true"]
        y_pred = merged[f"{var}_{model_tag}"]

        mask = y_true.notna() & y_pred.notna() & (y_pred != -1)
        y_true_m, y_pred_m = y_true[mask], y_pred[mask]

        print(f"=== {var} (n={mask.sum()}) ===")
        report_dict = classification_report(y_true_m, y_pred_m, zero_division=0, output_dict=True)
        print(classification_report(y_true_m, y_pred_m, zero_division=0))

        kappa = cohen_kappa_score(y_true_m, y_pred_m, weights=var_weights[var])
        alpha = krippendorff_alpha(y_true_m.values, y_pred_m.values, var_levels[var])
        print(f"Cohen's Kappa: {kappa:.3f}   Krippendorff's alpha ({var_levels[var]}): {alpha:.3f}\n")

        for class_label, metrics in report_dict.items():
            if isinstance(metrics, dict):  # skips "accuracy", which is a bare float
                all_rows.append({
                    "model": model_name, "variable": var, "n": mask.sum(),
                    "class": class_label,
                    "precision": metrics["precision"], "recall": metrics["recall"],
                    "f1_score": metrics["f1-score"], "support": metrics["support"],
                    "accuracy": round(report_dict["accuracy"], 3),
                    "cohens_kappa": round(kappa, 3),
                    "krippendorff_alpha": round(alpha, 3) if not np.isnan(alpha) else "n/a",
                })

report_df = pd.DataFrame(all_rows)
report_df.to_excel("data/comments/eval_report_coding_v10.xlsx", index=False)
print("Saved eval_report_coding_v10.xlsx")

Matched 135 of 135 rows


########## qwen3.8:27b-q4_K_M ##########
=== pers_exp (n=128) ===
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       112
           1       0.85      0.69      0.76        16

    accuracy                           0.95       128
   macro avg       0.90      0.83      0.86       128
weighted avg       0.94      0.95      0.94       128

Cohen's Kappa: 0.728   Krippendorff's alpha (nominal): 0.729

=== emot_exp (n=128) ===
              precision    recall  f1-score   support

           0       0.93      0.94      0.93        96
           1       0.81      0.78      0.79        32

    accuracy                           0.90       128
   macro avg       0.87      0.86      0.86       128
weighted avg       0.90      0.90      0.90       128

Cohen's Kappa: 0.726   Krippendorff's alpha (nominal): 0.727

=== pol_opin (n=128) ===
              precision    recall  f1-score   support

           0       0.93  

In [ ]:
merged.to_excel("data/comments/errors_qwen_v9.xlsx", index=False)
print("gespeichert")

gespeichert
